In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import subprocess
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ==========================================
# 1. HARDWARE CHECK & OPTIMIZATION
# ==========================================
print("--- SYSTEM HARDWARE CHECK ---")
try:
    # This prints the exact NVIDIA GPU model and memory limit to your console
    gpu_info = subprocess.check_output('nvidia-smi').decode('utf-8')
    print(gpu_info)
except Exception as e:
    print("NVIDIA-SMI not found. Are you sure the GPU accelerator is turned on?")

# Dynamically adjust training strategy based on available hardware
gpus = tf.config.list_physical_devices('GPU')
if len(gpus) > 1:
    print(f"🔥 {len(gpus)} GPUs detected! Engaging MirroredStrategy for parallel training.")
    strategy = tf.distribute.MirroredStrategy()
elif len(gpus) == 1:
    print("👍 1 GPU detected. Using default GPU strategy.")
    strategy = tf.distribute.get_strategy()
else:
    print("⚠️ WARNING: No GPU detected. CPU training will be painfully slow.")
    strategy = tf.distribute.get_strategy()

print(f"Total accelerators in sync: {strategy.num_replicas_in_sync}")

# ==========================================
# 2. DATASET SETUP (Vipool's Augmented Dataset)
# ==========================================
IMAGE_SIZE = (224, 224)
# Scale the batch size to max out the VRAM on however many GPUs you have
GLOBAL_BATCH_SIZE = 64 * strategy.num_replicas_in_sync 

# Exact paths based on the Kaggle dataset structure you linked
BASE_DIR = "/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VALID_DIR = os.path.join(BASE_DIR, "valid")

print(f"\nLoading data from: {BASE_DIR}")

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=GLOBAL_BATCH_SIZE,
    shuffle=True,
    seed=123
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    image_size=IMAGE_SIZE,
    batch_size=GLOBAL_BATCH_SIZE,
    shuffle=False,
    seed=123
)

class_names = train_dataset.class_names
print(f"Total classes found: {len(class_names)}")

# Save labels for Android
with open('/kaggle/working/labels.txt', 'w') as f:
    for name in class_names:
        f.write(name + '\n')

# ==========================================
# 3. PREPROCESSING PIPELINE
# ==========================================
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
  RandomFlip("horizontal_and_vertical"),
  RandomRotation(0.2), 
  RandomZoom(0.1),     
])

train_dataset = train_dataset.map(
    lambda x, y: (preprocess_input(data_augmentation(x, training=True)), y),
    num_parallel_calls=AUTOTUNE
).prefetch(buffer_size=AUTOTUNE)

val_dataset = val_dataset.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
).prefetch(buffer_size=AUTOTUNE)

# ==========================================
# 4. DISTRIBUTED MODEL BUILDING
# ==========================================
# We must build and compile the model INSIDE the strategy scope so the math replicates across all GPUs
with strategy.scope():
    base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False # Freeze base weights
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x) 
    outputs = Dense(len(class_names), activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# ==========================================
# 5. TRAINING WITH CALLBACKS
# ==========================================
# Checkpoints ensure we don't lose our progress if Kaggle disconnects
checkpoint = ModelCheckpoint('/kaggle/working/best_model.h5', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
# Early stopping prevents over-training if the model stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

print("\n--- INITIATING DISTRIBUTED TRAINING ---")
# Set epochs high; early stopping will catch it when it peaks
history = model.fit(train_dataset, validation_data=val_dataset, epochs=20, callbacks=[checkpoint, early_stop])

# ==========================================
# 6. EDGE QUANTIZATION (TFLITE)
# ==========================================
print("\n--- CONVERTING TO INT8 TFLITE ---")
# Ensure we are converting the absolute best version of the model
model.load_weights('/kaggle/working/best_model.h5')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] 
tflite_quant_model = converter.convert()

with open('/kaggle/working/model.tflite', 'wb') as f:
    f.write(tflite_quant_model)
    
print("\n✅ SUCCESS! 'model.tflite' and 'labels.txt' are saved in your /kaggle/working/ directory.")

2026-03-24 02:49:25.322331: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774320565.551253      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774320565.621505      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774320566.202908      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774320566.202954      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774320566.202956      25 computation_placer.cc:177] computation placer alr

--- SYSTEM HARDWARE CHECK ---
Tue Mar 24 02:49:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------

I0000 00:00:1774320595.294679      25 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774320595.300575      25 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 70295 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.
Total classes found: 38
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- INITIATING DISTRIBUTED TRAINING ---
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:loca

I0000 00:00:1774320671.938408      73 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774320671.939266      75 cuda_dnn.cc:529] Loaded cuDNN version 91002


...

550/550 ━━━━━━━━━━━━━━━━━━━━ 399s 722ms/step - accuracy: 0.9263 - loss: 0.2267 - val_accuracy: 0.9317 - val_loss: 0.2147
Epoch 9/20
549/550 ━━━━━━━━━━━━━━━━━━━━ 0s 681ms/step - accuracy: 0.9250 - loss: 0.2263
Epoch 9: val_accuracy improved from 0.93171 to 0.93603, saving model to /kaggle/working/best_model.h5


550/550 ━━━━━━━━━━━━━━━━━━━━ 401s 726ms/step - accuracy: 0.9250 - loss: 0.2262 - val_accuracy: 0.9360 - val_loss: 0.1975
Epoch 10/20
549/550 ━━━━━━━━━━━━━━━━━━━━ 0s 680ms/step - accuracy: 0.9273 - loss: 0.2165
Epoch 10: val_accuracy did not improve from 0.93603
550/550 ━━━━━━━━━━━━━━━━━━━━ 399s 722ms/step - accuracy: 0.9273 - loss: 0.2165 - val_accuracy: 0.9334 - val_loss: 0.2044
Epoch 11/20
549/550 ━━━━━━━━━━━━━━━━━━━━ 0s 657ms/step - accuracy: 0.9290 - loss: 0.2163
Epoch 11: val_accuracy did not improve from 0.93603
550/550 ━━━━━━━━━━━━━━━━━━━━ 387s 699ms/step - accuracy: 0.9290 - loss: 0.2163 - val_accuracy: 0.9309 - val_loss: 0.2110
Epoch 12/20
549/550 ━━━━━━━━━━━━━━━━━━━━ 0s 666ms/step - accuracy: 0.9287 - loss: 0.2146
Epoch 12: val_accuracy did not improve from 0.93603
550/550 ━━━━━━━━━━━━━━━━━━━━ 391s 708ms/step - accuracy: 0.9287 - loss: 0.2146 - val_accuracy: 0.9340 - val_loss: 0.2030
Epoch 12: early stopping
Restoring model weights from the end of the best epoch: 9.

--- CONV

INFO:tensorflow:Assets written to: /tmp/tmphvi1h6_k/assets


Saved artifact at '/tmp/tmphvi1h6_k'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_4')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  137624951242000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137622566618704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137622566619088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137622566617552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137622566618320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137622566617936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137622566621584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137622566622160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137622566620432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137622566619280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13762256662

W0000 00:00:1774325487.979581      25 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774325487.979636      25 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1774325488.126569      25 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
